# ESCI multi-task reranker inference

Load the trained **multi-task** ESCI reranker and score (query, product) pairs for a sample query from the test set. For each candidate, the model outputs:

- A **ranking score** (Task 1).
- A predicted **ESCI class** (Task 2).
- An **is-substitute probability** (Task 3).

This mirrors the behavior used by the FastAPI `/rerank` endpoint.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
CHECKPOINTS_DIR = ROOT / "checkpoints"

import pandas as pd

from src.data.load_data import ESCIDataLoader
from src.models.multi_task_reranker import load_multi_task_reranker

In [ ]:
# Load multi-task reranker and test data

model_path = CHECKPOINTS_DIR / "multi_task_reranker"  # matches configs/multi_task_reranker.yaml save_path
mt_reranker = load_multi_task_reranker(model_path=model_path)

test_path = DATA_DIR / "esci_test.parquet"
if test_path.exists():
    test_df = pd.read_parquet(test_path)
else:
    _, test_df = ESCIDataLoader(data_dir=DATA_DIR).prepare_train_test()

# Pick a sample query
sample = test_df.groupby("query_id").first().reset_index().iloc[0]
query = str(sample["query"])
qid = sample["query_id"]
print(f"Query: {query}")

In [ ]:
# Get all products for this query and rerank with multi-task outputs

rows = test_df[test_df["query_id"] == qid]
candidates = [(str(r["product_id"]), str(r["product_text"])) for _, r in rows.iterrows()]
ranked = mt_reranker.rerank(query, candidates, batch_size=16)

# Show top 5 with labels and substitute probabilities
pid_to_label = dict(zip(rows["product_id"].astype(str), rows["esci_label"]))
for rank, (pid, score, esci_class, sub_prob) in enumerate(ranked[:5], start=1):
    label = pid_to_label.get(pid, "?")
    text = next(t for p, t in candidates if p == pid)
    print(f"{rank}. [true={label} pred={esci_class}] score={score:.4f} sub_prob={sub_prob:.2f}")
    print(f"    {text[:200]}...")
    print()